# 03 — Evaluating the RFP automation engine

> **Applied Labs** · **Domain**: PE · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/11_proposal_rfp_automation/03_evaluate.ipynb)

**Prerequisites**:
- `02_build.ipynb` — a working RFP automation pipeline
- An OpenAI API key set as `OPENAI_API_KEY`

**What you will learn**:
- How to evaluate RFP answer quality using LLM-as-judge scoring
- How to measure retrieval relevance with precision@k metrics
- How to assess context adaptation depth and industry compliance
- How to audit consistency checking for false positives and negatives
- How to calculate cost, time savings, and ROI for RFP automation

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "numpy>=1.24.0"

import os, re, json, textwrap, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from collections import defaultdict
from openai import OpenAI

OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    print("⚠ OPENAI_API_KEY not set — using simulated eval scores")
    USE_LLM = False
else:
    client = OpenAI(api_key=OPENAI_API_KEY)
    USE_LLM = True

np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11
print("Setup complete ✓")

## Section 1 — Evaluation framework

We evaluate the RFP automation engine across **six dimensions**:

| Metric | What it measures | Target |
|---|---|---|
| Answer accuracy | Factual correctness of adapted answers | ≥4.0/5 |
| Retrieval relevance | Precision of past answer retrieval | P@3 ≥ 0.80 |
| Adaptation quality | Industry context properly incorporated | ≥4.0/5 |
| Consistency score | Contradictions detected and resolved | F1 ≥ 0.85 |
| Compliance accuracy | Correct requirement → capability mapping | ≥90% |
| Time savings | Hours saved per RFP vs manual | ≥80% reduction |

Each metric targets a different failure mode. Let's build the evaluation
harness.

In [ ]:
@dataclass
class EvalResult:
    """Result from evaluating a single answer."""
    question_id: str
    relevance: float      # 1-5
    completeness: float   # 1-5
    accuracy: float       # 1-5
    professionalism: float  # 1-5
    reasoning: str = ""

    @property
    def average(self) -> float:
        """Compute average score across all dimensions."""
        return np.mean([self.relevance, self.completeness,
                        self.accuracy, self.professionalism])

@dataclass
class EvalSuite:
    """Complete evaluation suite results."""
    answer_evals: List[EvalResult] = field(default_factory=list)
    retrieval_metrics: Dict[str, float] = field(default_factory=dict)
    adaptation_scores: Dict[str, float] = field(default_factory=dict)
    consistency_audit: Dict[str, Any] = field(default_factory=dict)
    efficiency_metrics: Dict[str, float] = field(default_factory=dict)

# Sample adapted answers from the build notebook (simulated)
SAMPLE_ANSWERS: Dict[str, Dict[str, str]] = {
    "Q01": {"question": "Describe your data encryption approach for both data at rest and data in transit.",
            "category": "security",
            "adapted": "In alignment with healthcare regulatory requirements including HIPAA, we use AES-256 encryption for all data at rest, with keys managed through AWS KMS with automatic 90-day rotation. All data in transit is protected via TLS 1.3. PHI is encrypted with dedicated encryption keys separate from general data, ensuring maximum protection of sensitive health information.",
            "retrieval_score": 0.89},
    "Q02": {"question": "What access control and authentication mechanisms does your platform support?",
            "category": "security",
            "adapted": "Our platform implements role-based access control (RBAC) with SSO via SAML 2.0 and OIDC, meeting enterprise healthcare authentication standards. MFA is required for all administrative accounts. Granular permission sets allow control at the resource level, with complete audit logging of all access events for HIPAA compliance.",
            "retrieval_score": 0.87},
    "Q03": {"question": "Detail your vulnerability management and penetration testing program.",
            "category": "security",
            "adapted": "We conduct quarterly penetration testing through independent third-party security firms, with continuous SAST/DAST scanning integrated into our CI/CD pipeline. Critical vulnerabilities are patched within 24 hours. All findings are documented and tracked to resolution, with reports available for healthcare compliance audits.",
            "retrieval_score": 0.91},
    "Q04": {"question": "What compliance certifications does your organization currently hold?",
            "category": "compliance",
            "adapted": "We maintain SOC 2 Type II, ISO 27001, and HITRUST CSF r2 certifications, with FedRAMP Moderate authorization in progress. Annual third-party audits are conducted, with reports available under NDA. Our HITRUST certification specifically demonstrates our commitment to healthcare data protection standards.",
            "retrieval_score": 0.93},
    "Q05": {"question": "Describe your HIPAA compliance program, including BAA execution and PHI safeguards.",
            "category": "compliance",
            "adapted": "We execute Business Associate Agreements (BAAs) with all covered entities. All Protected Health Information (PHI) is encrypted at rest and in transit with dedicated encryption keys. We maintain comprehensive access audit logs, conduct annual HIPAA risk assessments, and provide breach notification within the required 60-day window.",
            "retrieval_score": 0.95},
    "Q06": {"question": "How do you handle audit requests and what compliance documentation is available?",
            "category": "compliance",
            "adapted": "SOC 2 Type II reports are available within 5 business days under NDA. We support on-site audit visits for enterprise healthcare customers. Our dedicated compliance team provides HIPAA compliance documentation, BAA copies, and security assessment questionnaires upon request.",
            "retrieval_score": 0.88},
    "Q07": {"question": "Describe your system architecture and deployment model.",
            "category": "technical",
            "adapted": "Cloud-native microservices architecture deployed on Kubernetes across 3 AWS regions. Event-driven processing with async message queues ensures reliability. All healthcare workloads run in HIPAA-eligible AWS services with dedicated VPCs and encryption at every layer.",
            "retrieval_score": 0.86},
    "Q08": {"question": "What is your disaster recovery plan, including RPO and RTO targets?",
            "category": "technical",
            "adapted": "Active-passive disaster recovery with automatic failover across geo-redundant regions. RPO: 15 minutes, RTO: 1 hour. Quarterly DR drills with documented results available for audit. Healthcare data recovery procedures include PHI integrity verification.",
            "retrieval_score": 0.92},
    "Q09": {"question": "What uptime SLA do you guarantee and what remedies are available?",
            "category": "technical",
            "adapted": "We guarantee 99.9% uptime SLA with financially-backed service credits: 10% credit at 99.5-99.9% uptime, 25% credit below 99.5%. Real-time status page provides transparency. Healthcare-critical systems receive priority incident response.",
            "retrieval_score": 0.90},
    "Q10": {"question": "Describe your pricing model, including per-user costs and volume discounts.",
            "category": "pricing",
            "adapted": "Per-seat monthly pricing: Standard at $25/user/month, Professional at $50/user/month, Enterprise at custom pricing. Volume discounts available at 100+ and 500+ seats. Healthcare-specific compliance features included in Professional and Enterprise tiers at no additional cost.",
            "retrieval_score": 0.85},
    "Q11": {"question": "What are the implementation costs and typical timeline?",
            "category": "pricing",
            "adapted": "Standard implementation included at no additional cost. Custom integrations, including EHR connectivity, billed at $200/hour. Typical enterprise healthcare implementation: 8-10 weeks, $20-30K, including HIPAA compliance configuration and staff training.",
            "retrieval_score": 0.84},
    "Q12": {"question": "What support tiers are available and response time commitments?",
            "category": "support",
            "adapted": "Standard: email support with 24-hour response. Premium: phone and email with 4-hour response. Enterprise: dedicated Customer Success Manager, 1-hour critical issue response, and quarterly business reviews. Healthcare customers receive priority routing for PHI-related issues.",
            "retrieval_score": 0.91},
    "Q13": {"question": "Describe your API capabilities and rate limits.",
            "category": "integration",
            "adapted": "RESTful API with OpenAPI 3.0 specification and interactive documentation. Rate limits: 1,000 requests/min (standard), 10,000 requests/min (enterprise). SDKs available for Python, JavaScript, and Java. Healthcare-specific endpoints for HL7 FHIR R4 data exchange.",
            "retrieval_score": 0.88},
    "Q14": {"question": "Can your platform integrate with major EHR systems?",
            "category": "integration",
            "adapted": "Yes. We support HL7 FHIR R4 for EHR integration with pre-built connectors for Epic, Cerner, and Allscripts. Custom FHIR resource mapping is available for specialized healthcare data workflows. Integration typically completed within 2-4 weeks.",
            "retrieval_score": 0.94},
    "Q15": {"question": "Provide customer references in the healthcare industry.",
            "category": "references",
            "adapted": "We are pleased to provide the following healthcare references: (1) Regional Health System — 500-bed hospital network, deployed platform-wide for 3 years; (2) CareFirst Clinic Network — 45 outpatient clinics, 98% user satisfaction score; (3) MedTech Solutions — healthcare SaaS company, integrated via FHIR API.",
            "retrieval_score": 0.82},
}

print(f"Loaded {len(SAMPLE_ANSWERS)} sample answers for evaluation")
categories = defaultdict(int)
for a in SAMPLE_ANSWERS.values():
    categories[a["category"]] += 1
print(f"Categories: {dict(categories)}")

## Section 2 — Answer quality (LLM-as-judge)

We use **LLM-as-judge** to score each adapted answer on four dimensions:
relevance (1–5), completeness (1–5), accuracy (1–5), and
professionalism (1–5). The judge receives the original question and the
adapted answer.

In [ ]:
JUDGE_SYSTEM_PROMPT: str = """You are an expert RFP evaluator for healthcare IT procurement.
Score the ANSWER to the given QUESTION on four dimensions (1-5 each):

1. RELEVANCE: Does the answer directly address the question?
2. COMPLETENESS: Does it cover all aspects of the question?
3. ACCURACY: Are factual claims credible and consistent?
4. PROFESSIONALISM: Is the tone appropriate for enterprise healthcare?

Return ONLY a JSON object:
{"relevance": N, "completeness": N, "accuracy": N, "professionalism": N, "reasoning": "brief explanation"}
"""

def judge_answer_llm(question: str, answer: str) -> EvalResult:
    """Score an answer using LLM-as-judge.

    Falls back to heuristic scoring if API key not available.
    """
    if not USE_LLM:
        return judge_answer_heuristic(question, answer)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": f"QUESTION: {question}\n\nANSWER: {answer}"},
        ],
        temperature=0.1,
        max_tokens=200,
        response_format={"type": "json_object"},
    )
    scores = json.loads(response.choices[0].message.content)
    return EvalResult(
        question_id="",
        relevance=float(scores.get("relevance", 3)),
        completeness=float(scores.get("completeness", 3)),
        accuracy=float(scores.get("accuracy", 3)),
        professionalism=float(scores.get("professionalism", 3)),
        reasoning=scores.get("reasoning", ""),
    )

def judge_answer_heuristic(question: str, answer: str) -> EvalResult:
    """Heuristic scoring based on answer properties.

    Evaluates length, keyword coverage, and structure.
    """
    q_words = set(re.findall(r"\w{4,}", question.lower()))
    a_words = set(re.findall(r"\w{4,}", answer.lower()))
    overlap = len(q_words & a_words) / max(len(q_words), 1)

    relevance = min(5.0, 2.0 + overlap * 4)
    completeness = min(5.0, 2.5 + (len(answer) / 200))
    accuracy = 4.0 + np.random.uniform(-0.5, 0.5)
    professionalism = 4.2 + np.random.uniform(-0.3, 0.3)

    return EvalResult(
        question_id="",
        relevance=round(min(relevance, 5.0), 1),
        completeness=round(min(completeness, 5.0), 1),
        accuracy=round(min(accuracy, 5.0), 1),
        professionalism=round(min(professionalism, 5.0), 1),
        reasoning="Heuristic scoring (no API key)",
    )

# Evaluate all 15 answers
eval_results: List[EvalResult] = []

print("LLM-as-judge evaluation:\n")
for qid, ans in SAMPLE_ANSWERS.items():
    result = judge_answer_llm(ans["question"], ans["adapted"])
    result.question_id = qid
    eval_results.append(result)
    print(f"  {qid}: rel={result.relevance:.1f} comp={result.completeness:.1f} "
          f"acc={result.accuracy:.1f} prof={result.professionalism:.1f} "
          f"avg={result.average:.2f}")

overall_avg = np.mean([r.average for r in eval_results])
print(f"\nOverall average score: {overall_avg:.2f}/5.0")
print(f"Target: ≥4.0/5.0 → {"✓ PASS" if overall_avg >= 4.0 else "✗ BELOW TARGET"}")

## Section 3 — Retrieval relevance

Are the right past answers being retrieved? We measure **precision@k=3**:
of the top-3 retrieved answers, how many are actually relevant to the
question? We also analyze retrieval errors — false positives (irrelevant
answers ranked high) and misses.

In [ ]:
# Ground truth: which topics should match each question
GROUND_TRUTH: Dict[str, List[str]] = {
    "Q01": ["security"],
    "Q02": ["security"],
    "Q03": ["security"],
    "Q04": ["compliance"],
    "Q05": ["compliance"],
    "Q06": ["compliance"],
    "Q07": ["technical"],
    "Q08": ["technical"],
    "Q09": ["technical"],
    "Q10": ["pricing"],
    "Q11": ["pricing"],
    "Q12": ["support"],
    "Q13": ["integration"],
    "Q14": ["integration"],
    "Q15": ["integration", "support"],  # references can match multiple
}

# Simulated retrieval results (top-3 topics per question)
RETRIEVAL_RESULTS: Dict[str, List[Dict[str, Any]]] = {}
for qid, expected_topics in GROUND_TRUTH.items():
    results: List[Dict[str, Any]] = []
    primary = expected_topics[0]
    # Simulate: top-1 usually correct, top-2/3 sometimes drift
    topics_pool = ["security", "compliance", "technical", "pricing", "support", "integration"]
    results.append({"topic": primary, "score": 0.85 + np.random.uniform(0, 0.1)})
    if np.random.random() > 0.2:  # 80% top-2 correct
        results.append({"topic": primary, "score": 0.70 + np.random.uniform(0, 0.1)})
    else:
        wrong = np.random.choice([t for t in topics_pool if t != primary])
        results.append({"topic": wrong, "score": 0.65 + np.random.uniform(0, 0.1)})
    if np.random.random() > 0.4:  # 60% top-3 correct
        results.append({"topic": primary, "score": 0.55 + np.random.uniform(0, 0.1)})
    else:
        wrong = np.random.choice([t for t in topics_pool if t != primary])
        results.append({"topic": wrong, "score": 0.50 + np.random.uniform(0, 0.1)})
    RETRIEVAL_RESULTS[qid] = results

def precision_at_k(retrieved: List[Dict[str, Any]],
                   relevant_topics: List[str], k: int = 3) -> float:
    """Compute precision@k for a single query.

    Args:
        retrieved: List of retrieved results with topic field.
        relevant_topics: List of ground-truth relevant topics.
        k: Cutoff for precision calculation.

    Returns:
        Precision@k as float 0-1.
    """
    top_k = retrieved[:k]
    relevant_count = sum(1 for r in top_k if r["topic"] in relevant_topics)
    return relevant_count / k

# Calculate P@3 for each question
precision_scores: Dict[str, float] = {}
print("Retrieval Precision@3 per question:\n")
for qid in GROUND_TRUTH:
    p3 = precision_at_k(RETRIEVAL_RESULTS[qid], GROUND_TRUTH[qid], k=3)
    precision_scores[qid] = p3
    topics = [r["topic"] for r in RETRIEVAL_RESULTS[qid]]
    status = "✓" if p3 >= 0.67 else "✗"
    print(f"  {qid}: P@3={p3:.2f} {status}  retrieved={topics}  expected={GROUND_TRUTH[qid]}")

avg_p3 = np.mean(list(precision_scores.values()))
print(f"\nAverage P@3: {avg_p3:.3f}")
print(f"Target: ≥0.80 → {"✓ PASS" if avg_p3 >= 0.80 else "✗ BELOW TARGET"}")

# Error analysis
low_precision = {k: v for k, v in precision_scores.items() if v < 0.67}
if low_precision:
    print(f"\nLow-precision questions ({len(low_precision)}):")
    for qid, p in low_precision.items():
        print(f"  {qid}: P@3={p:.2f} — topic drift in retrieval")

## Section 4 — Adaptation quality

Is the healthcare/HIPAA context properly incorporated? We check:

1. **Industry term presence** — Do healthcare-specific terms appear?
2. **Requirement coverage** — Are HIPAA, PHI, BAA mentioned where needed?
3. **Adaptation depth** — How much was the answer actually changed?

This is a *content-level* evaluation, not just a style check.

In [ ]:
HEALTHCARE_TERMS: List[str] = [
    "HIPAA", "PHI", "BAA", "ePHI", "covered entity",
    "healthcare", "patient", "clinical", "EHR", "HL7", "FHIR",
    "HITRUST", "health information",
]

def score_adaptation(question: str, answer: str,
                     required_terms: List[str],
                     category: str) -> Dict[str, Any]:
    """Score how well an answer is adapted to healthcare context.

    Returns dict with term coverage, adaptation score, and details.
    """
    answer_lower = answer.lower()
    found_terms = [t for t in required_terms if t.lower() in answer_lower]
    missing_terms = [t for t in required_terms if t.lower() not in answer_lower]

    # Category-specific required terms
    category_requirements: Dict[str, List[str]] = {
        "security": ["HIPAA", "PHI", "encryption"],
        "compliance": ["HIPAA", "BAA", "audit"],
        "technical": ["healthcare", "HIPAA"],
        "pricing": ["healthcare"],
        "support": ["healthcare", "PHI"],
        "integration": ["EHR", "FHIR", "HL7"],
    }

    cat_reqs = category_requirements.get(category, [])
    cat_found = [t for t in cat_reqs if t.lower() in answer_lower]
    cat_coverage = len(cat_found) / max(len(cat_reqs), 1)

    overall_coverage = len(found_terms) / max(len(required_terms), 1)
    adaptation_score = (cat_coverage * 0.7 + overall_coverage * 0.3) * 5

    return {
        "terms_found": found_terms,
        "terms_missing": missing_terms,
        "category_coverage": round(cat_coverage, 2),
        "overall_coverage": round(overall_coverage, 2),
        "adaptation_score": round(min(adaptation_score, 5.0), 2),
    }

# Evaluate adaptation for all answers
adaptation_results: List[Dict[str, Any]] = []

print("Adaptation quality analysis:\n")
for qid, ans in SAMPLE_ANSWERS.items():
    result = score_adaptation(
        ans["question"], ans["adapted"],
        HEALTHCARE_TERMS, ans["category"]
    )
    result["question_id"] = qid
    result["category"] = ans["category"]
    adaptation_results.append(result)
    status = "✓" if result["adaptation_score"] >= 3.0 else "✗"
    print(f"  {qid} ({ans['category']:>11}): score={result['adaptation_score']:.1f}/5 "
          f"cat_cov={result['category_coverage']:.0%} {status}")
    if result["terms_missing"] and len(result["terms_missing"]) <= 3:
        print(f"    Missing: {result['terms_missing']}")

avg_adapt = np.mean([r["adaptation_score"] for r in adaptation_results])
print(f"\nAverage adaptation score: {avg_adapt:.2f}/5.0")
print(f"Target: ≥4.0/5.0 → {"✓ PASS" if avg_adapt >= 4.0 else "→ review needed"}")

## Section 5 — Consistency analysis

We audit the consistency checker's performance:
- **True positives**: Real contradictions correctly flagged
- **False positives**: Non-contradictions incorrectly flagged
- **False negatives**: Real contradictions missed

We inject known contradictions to measure detection accuracy.

In [ ]:
# Test cases: sections with known consistency properties
CONSISTENCY_TEST_CASES: List[Dict[str, Any]] = [
    {
        "name": "Uptime contradiction",
        "sections": {
            "Q08": "Our platform maintains 99.9% uptime with 1 hour RTO.",
            "Q09": "We guarantee 99.99% availability backed by SLA credits.",
        },
        "expected_contradictions": 1,
        "expected_type": "uptime",
    },
    {
        "name": "RTO contradiction",
        "sections": {
            "Q08": "Our RTO target is 1 hour for disaster recovery.",
            "Q12": "Critical issues are resolved with 2 hour RTO guarantee.",
        },
        "expected_contradictions": 1,
        "expected_type": "rto",
    },
    {
        "name": "Consistent sections",
        "sections": {
            "Q01": "AES-256 encryption with SOC 2 Type II certification.",
            "Q04": "We hold SOC 2 Type II and ISO 27001 certifications.",
        },
        "expected_contradictions": 0,
        "expected_type": None,
    },
    {
        "name": "Pricing contradiction",
        "sections": {
            "Q10": "Standard pricing is $25/user/month.",
            "Q11": "Base platform cost starts at $30/user/month.",
        },
        "expected_contradictions": 1,
        "expected_type": "pricing",
    },
    {
        "name": "Response time contradiction",
        "sections": {
            "Q09": "Critical incidents receive 15 minute response.",
            "Q12": "Enterprise tier: 1 hour response for critical issues.",
        },
        "expected_contradictions": 1,
        "expected_type": "response_time",
    },
]

# Consistency checker (reused from build notebook)
CLAIM_PATTERNS: Dict[str, re.Pattern] = {
    "uptime": re.compile(r"(\d{2,3}\.\d+)\s*%\s*(?:uptime|availability)", re.I),
    "rto": re.compile(r"(\d+)\s*-?\s*(?:hour|hr|minute|min)\s*RTO", re.I),
    "rpo": re.compile(r"(\d+)\s*-?\s*(?:hour|hr|minute|min)\s*RPO", re.I),
    "response_time": re.compile(r"(\d+)\s*(?:hour|hr|minute|min)\s*response", re.I),
    "pricing": re.compile(r"\$(\d+(?:,\d{3})*(?:\.\d{2})?)\s*/", re.I),
}

def check_consistency(sections: Dict[str, str]) -> List[Dict[str, Any]]:
    """Check for contradictions in proposal sections.

    Returns list of detected contradictions.
    """
    all_claims: List[Dict[str, str]] = []
    for sid, text in sections.items():
        for ctype, pattern in CLAIM_PATTERNS.items():
            for m in pattern.finditer(text):
                all_claims.append({"section": sid, "type": ctype,
                                   "value": m.group(1), "raw": m.group(0)})

    grouped: Dict[str, List[Dict[str, str]]] = defaultdict(list)
    for c in all_claims:
        grouped[c["type"]].append(c)

    contradictions: List[Dict[str, Any]] = []
    for ctype, claims in grouped.items():
        values = set(c["value"] for c in claims)
        if len(values) > 1:
            contradictions.append({"type": ctype, "values": list(values),
                                   "sections": [c["section"] for c in claims]})
    return contradictions

# Run tests
tp, fp, fn = 0, 0, 0
print("Consistency checker audit:\n")
for tc in CONSISTENCY_TEST_CASES:
    detected = check_consistency(tc["sections"])
    n_detected = len(detected)
    expected = tc["expected_contradictions"]

    if expected > 0 and n_detected > 0:
        tp += 1
        status = "✓ TP"
    elif expected == 0 and n_detected == 0:
        status = "✓ TN"
    elif expected == 0 and n_detected > 0:
        fp += 1
        status = "✗ FP"
    else:
        fn += 1
        status = "✗ FN"

    print(f"  {tc['name']:30s} expected={expected} detected={n_detected} → {status}")

precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-9)

print(f"\nConsistency checker metrics:")
print(f"  TP={tp}  FP={fp}  FN={fn}")
print(f"  Precision: {precision:.2f}")
print(f"  Recall:    {recall:.2f}")
print(f"  F1:        {f1:.2f}")
print(f"  Target F1 ≥0.85 → {"✓ PASS" if f1 >= 0.85 else "✗ BELOW TARGET"}")

## Section 6 — Cost and efficiency analysis

The bottom line: how much time and money does automation save?
We model three scenarios and visualize the ROI.

In [ ]:
def calculate_efficiency(
    questions: int = 15,
    manual_hours_per_q: float = 1.5,
    auto_minutes_per_q: float = 2.0,
    review_minutes_per_q: float = 5.0,
    hourly_rate: float = 175.0,
    api_cost_per_q: float = 0.02,
) -> Dict[str, float]:
    """Calculate time and cost savings for RFP automation.

    Args:
        questions: Number of RFP questions.
        manual_hours_per_q: Hours per question (manual).
        auto_minutes_per_q: Minutes per question (automated).
        review_minutes_per_q: Minutes per question (human review).
        hourly_rate: Cost per hour of expert time.
        api_cost_per_q: LLM API cost per question.

    Returns:
        Dict with time, cost, and savings metrics.
    """
    manual_hours = questions * manual_hours_per_q
    auto_hours = questions * (auto_minutes_per_q + review_minutes_per_q) / 60

    manual_cost = manual_hours * hourly_rate
    auto_cost = auto_hours * hourly_rate + questions * api_cost_per_q

    return {
        "questions": questions,
        "manual_hours": round(manual_hours, 1),
        "auto_hours": round(auto_hours, 1),
        "time_saved_pct": round((1 - auto_hours / manual_hours) * 100, 1),
        "manual_cost": round(manual_cost, 2),
        "auto_cost": round(auto_cost, 2),
        "cost_saved": round(manual_cost - auto_cost, 2),
        "cost_saved_pct": round((1 - auto_cost / manual_cost) * 100, 1),
    }

# Single RFP efficiency
single = calculate_efficiency()
print("Single RFP (15 questions):")
print(f"  Manual: {single['manual_hours']} hrs (${single['manual_cost']:,.0f})")
print(f"  Auto:   {single['auto_hours']} hrs (${single['auto_cost']:,.0f})")
print(f"  Time saved: {single['time_saved_pct']:.0f}%")
print(f"  Cost saved: ${single['cost_saved']:,.0f} ({single['cost_saved_pct']:.0f}%)")

# Annual scenarios
scenarios = [
    ("Small team",  50, 15),
    ("Mid-market", 150, 20),
    ("Enterprise", 400, 25),
]

annual_data: List[Dict[str, Any]] = []
print(f"\nAnnual ROI by company size:")
print(f"{'Scenario':>15} {'RFPs/yr':>8} {'Manual Cost':>12} {'Auto Cost':>12} {'Savings':>12} {'ROI':>8}")
print("-" * 72)
for name, rfps, avg_questions in scenarios:
    eff = calculate_efficiency(questions=avg_questions)
    annual_manual = eff["manual_cost"] * rfps
    annual_auto = eff["auto_cost"] * rfps + 50_000  # platform cost
    annual_savings = annual_manual - annual_auto
    roi = (annual_savings / annual_auto) * 100
    print(f"{name:>15} {rfps:>8} ${annual_manual:>10,.0f} ${annual_auto:>10,.0f} "
          f"${annual_savings:>10,.0f} {roi:>6.0f}%")
    annual_data.append({
        "name": name, "rfps": rfps,
        "manual": annual_manual, "auto": annual_auto,
        "savings": annual_savings, "roi": roi,
    })

In [ ]:
# Visualization: evaluation summary
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Answer quality scores
categories_order = ["relevance", "completeness", "accuracy", "professionalism"]
avg_scores = {
    "relevance": np.mean([r.relevance for r in eval_results]),
    "completeness": np.mean([r.completeness for r in eval_results]),
    "accuracy": np.mean([r.accuracy for r in eval_results]),
    "professionalism": np.mean([r.professionalism for r in eval_results]),
}
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]
bars = axes[0].bar(categories_order, [avg_scores[c] for c in categories_order],
                   color=colors, edgecolor="white")
axes[0].axhline(y=4.0, color="red", linestyle="--", alpha=0.7, label="Target (4.0)")
axes[0].set_ylim(0, 5.2)
axes[0].set_title("Answer Quality Scores")
axes[0].set_ylabel("Score (1-5)")
axes[0].legend()
for bar, val in zip(bars, [avg_scores[c] for c in categories_order]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f"{val:.2f}", ha="center", fontsize=9)

# Plot 2: Annual savings by scenario
scenario_names = [d["name"] for d in annual_data]
manual_costs = [d["manual"] / 1_000_000 for d in annual_data]
auto_costs = [d["auto"] / 1_000_000 for d in annual_data]
x = np.arange(len(scenario_names))
axes[1].bar(x - 0.2, manual_costs, 0.35, label="Manual", color="#EF5350")
axes[1].bar(x + 0.2, auto_costs, 0.35, label="Automated", color="#66BB6A")
axes[1].set_xticks(x)
axes[1].set_xticklabels(scenario_names)
axes[1].set_title("Annual Cost Comparison")
axes[1].set_ylabel("Cost ($M)")
axes[1].legend()

# Plot 3: Retrieval P@3 distribution
p3_values = list(precision_scores.values())
axes[2].hist(p3_values, bins=[0, 0.33, 0.67, 1.01], color="#42A5F5",
             edgecolor="white", rwidth=0.85)
axes[2].axvline(x=0.80, color="red", linestyle="--", alpha=0.7, label="Target (0.80)")
axes[2].set_title("Retrieval P@3 Distribution")
axes[2].set_xlabel("Precision@3")
axes[2].set_ylabel("Number of questions")
axes[2].legend()

plt.tight_layout()
plt.show()
print("✓ Evaluation visualizations rendered")

## Exercises

1. **Improve retrieval with re-ranking** — Implement a cross-encoder
   re-ranker on top of the bi-encoder retrieval. Measure whether P@3
   improves compared to the baseline.

2. **Multi-judge ensemble** — Run LLM-as-judge with 3 different
   temperature settings (0.0, 0.3, 0.7) and compute inter-judge agreement
   (Cohen's kappa). Does ensembling improve evaluation reliability?

3. **Win rate simulation** — Given quality scores, model the probability
   of winning an RFP at different score thresholds. What minimum average
   score correlates with a 50% win rate?

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | LLM-as-judge provides scalable answer quality evaluation across 4 dimensions |
| 2 | Retrieval P@3 measures whether the right past answers are surfaced |
| 3 | Adaptation quality requires domain-specific term coverage analysis |
| 4 | Consistency checking achieves high F1 on structured claim types (uptime, pricing) |
| 5 | Automation reduces RFP response time by 80-90% and cost by 70-85% |
| 6 | Enterprise customers with 400+ RFPs/year see $1-3 M annual savings |
| 7 | Human review remains essential — automation assists, not replaces |

## What's Next

Continue to **Lab 12 — Enterprise Search** to explore how similar retrieval
and generation patterns apply to organization-wide knowledge search.

---

## References

1. Zheng, L. et al., *Judging LLM-as-a-Judge*, NeurIPS 2023 — <https://arxiv.org/abs/2306.05685>
2. RFPIO/Responsive, *State of Strategic Response Management*, 2023 — <https://www.responsive.io/>
3. Loopio, *2023 RFP Response Trends* — <https://loopio.com/>
4. APMP, *Proposal Management Best Practices* — <https://www.apmp.org/>